In [1]:
import numpy as np

S = QKᵀ / √d <br>
P = softmax(S) (row-wise) <br>
O = PV


DO = G = $\partial$ L/ $\partial$ O

1 Derive $\partial$ V: <br>
    $\partial$ dv = $\partial$ L / $\partial$ O * $\partial$ O / $\partial$ V <br>
    as we Know $\partial$ L / $\partial$ O   = G <br>
    and $\partial$ O / $\partial$ V = P <br>
    so $\partial$ V = G Pᵀ <br>

2 Derive $\partial$ P: <br>
    $\partial$ dv = $\partial$ L / $\partial$ O * $\partial$ O / $\partial$ P <br>
    as we Know $\partial$ L / $\partial$ O   = G <br>
    and $\partial$ O / $\partial$ P = V <br>
    so $\partial$ V = G Vᵀ <br>

3 derive  $\partial$ S: <br>

Now $\partial S$:

$$P = \frac{e^{S_i}}{\sum_j e^{S_j}}$$

Kronecker delta form:

$$\frac{\partial P_i}{\partial S_j} = P_i\,(\delta_{ij} - P_j)$$
$$\delta_{ij} = \begin{cases} 1 & i = j \\ 0 & i \neq j \end{cases}$$



$$\frac{\partial L}{\partial S} = \frac{\partial L}{\partial P}\,\frac{\partial P}{\partial S}$$

for $p_i$: $\quad g_i = \dfrac{\partial L}{\partial p_i}$

$$\frac{\partial L}{\partial s_j} = \sum_i \frac{\partial L}{\partial p_i}\,\frac{\partial p_i}{\partial s_j}$$
$$\frac{\partial L}{\partial s_j} = \sum_i g_i\, p_i\,(\delta_{ij} - p_j)$$
$$\frac{\partial L}{\partial s_j} = \sum_i g_i\, p_i\, \delta_{ij} - \sum_i g_i\, p_i\, p_j$$
$$= \sum_i g_i\, p_i\, \delta_{ij} - p_j \sum_i g_i\, p_i$$
$$= g_j\, p_j - p_j \sum_i g_i\, p_i$$
$$\frac{\partial L}{\partial s_j} = p_j\left(g_j - \sum_i g_i\, p_i\right) \qquad ; \quad g = \partial P$$
$$\partial S = P \times \left[\partial P - \sum_i (P \times \partial P)_i\right]$$


4 derive $\partial$ Q 
$$\partial Q = \frac{\partial L}{\partial Q} \qquad\qquad \partial K = \frac{\partial L}{\partial K}$$
$$S = QK^T$$
$$S_{ij} = \sum_m Q_{im}\, K_{jm}$$
$$\frac{\partial L}{\partial Q_{im}} = \sum_j \frac{\partial L}{\partial S_{ij}}\,\frac{\partial S_{ij}}{\partial Q_{im}}$$
$$\frac{\partial S_{ij}}{\partial Q_{im}} = K_{jm}$$

we get

$$\frac{\partial L}{\partial Q_{im}} = \sum_j \partial S_{ij}\, K_{jm}$$
$$\boxed{\partial Q = \frac{\partial S^T \times K}{\sqrt{d}}}$$

for $\partial$ K same formula
$$\boxed{\partial K = \frac{\partial S^T\, Q}{\sqrt{d}}}$$

In [4]:
import numpy as np

def softmax(S):
    S = S - S.max(axis=-1, keepdims=True)      # stability
    E = np.exp(S)
    return E / E.sum(axis=-1, keepdims=True)

def attn_forward(Q, K, V, causal=False):
    d = Q.shape[-1]
    S = Q @ K.T / np.sqrt(d)
    if causal:
        S = np.where(np.triu(np.ones(S.shape, bool), k=1), -np.inf, S)
    P = softmax(S)
    return P @ V, (Q, K, V, P)

def attn_backward(dO, cache):
    Q, K, V, P = cache
    d = Q.shape[-1]
    dV = P.T @ dO
    dP = dO @ V.T
    dS = P * (dP - (P * dP).sum(axis=-1, keepdims=True))
    return dS @ K / np.sqrt(d), dS.T @ Q / np.sqrt(d), dV   # dQ, dK, dV

def num_grad(f, x, h):
    g = np.zeros_like(x)
    for i in np.ndindex(*x.shape):
        old = x[i]
        x[i] = old + h; fp = f()
        x[i] = old - h; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * h)
    return g

def rel_err(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b))

In [ ]:
import numpy as np

np.random.seed(42)

n, d_k, d_v = 3, 4, 2

Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

# A fixed upstream gradient
dO = np.random.randn(n, d_v)

# Analytical gradients
O, cache = attn_forward(Q, K, V)
dQ, dK, dV = attn_backward(dO, cache)

# Scalar objective whose gradient we want
def loss():
    O, _ = attn_forward(Q, K, V)
    return np.sum(O * dO)

# Numerical gradients
h = 1e-5
num_dQ = num_grad(loss, Q, h)
num_dK = num_grad(loss, K, h)
num_dV = num_grad(loss, V, h)

# Compare
print("dQ error:", rel_err(dQ, num_dQ))
print("dK error:", rel_err(dK, num_dK))
print("dV error:", rel_err(dV, num_dV))

dQ error: 1.6102773378766916e-11
dK error: 5.542699511411362e-11
dV error: 5.526774661881642e-12


# 2 State the step size, precision, and acceptance threshold.

Step size: \(h = 10^{-5}\) (1e-5), used for central finite differences.

Precision: Use float64 (double precision) for numerical gradient checking.

Acceptance threshold: Accept gradients when the relative error is less than \(10^{-5}\), i.e. rel_err < 1e-5.


# 3 Identify the forward-pass numerical stability issue and explain its fix.

The numerical stability issue occurs in the softmax function, <br> where calculating exponentials of large attention scores can cause overflow and produce inf or NaN values.

Fix: Subtract the maximum score from each row before exponentiation:
S = S - S.max(axis=-1, keepdims=True)
E = np.exp(S)return E / E.sum(axis=-1, keepdims=True)


# 4 Explain why attention uses the scaling factor 1/√d.

The attention mechanism uses the scaling factor 1/ $\sqrt{d_k}$  (where $\sqrt{dk}$ is the key dimension <br> to control the variance of the dot products. As vector dimensions grow larger, raw dot product values increase in magnitude,<br> which pushes the softmax function into regions of near-zero gradients. Scaling by $\sqrt{d_k}$  ) stabilizes these values and keeps training gradients healthy.


# 5 How the backward pass changes with a causal mask
it doesn’t change.

A causal mask means each word can only look at itself and earlier words, never at future ones. In the forward pass, we force the “future” scores to −∞, so after softmax their probabilities become exactly 0.


dS = P ∘ (dP − D)

For a masked position, P = 0, so dS = 0 × (something) = 0. The masked positions automatically get zero gradient,
they had no effect on the output, so changing them can’t change the loss.

Everything else (dV, dQ, dK) uses the same formulas as before.

# 6 Analyze memory cost as sequence length n grows and explain how FlashAttention avoids it.
Standard attention builds two big tables: the score table S and the probability table P. Each has one entry for every pair of positions, so its size is n × n. 
so its memory cost grows as O(n^2). If the sequence length doubles, memory usage for these matrices becomes approximately four times 
larger.

The FlashAttention 

Never build the full n × n table. Instead:

Work in small tiles. Cut Q, K and V into small blocks that fit in the GPU’s fast on-chip memory. Compute one small tile of scores, use it right away, and throw it away.

Extra memory		O(n)
